-=============================================================================
SCD TYPE 2 - COMPLETE IMPLEMENTATION (DATABRICKS / PYSPARK NOTEBOOK)
=============================================================================
Run each cell one by one in your notebook.
You can update the source data in CELL 4 to test different scenarios.
=============================================================================

In [0]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_date, monotonically_increasing_id,
    row_number, to_date
)
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType
from delta.tables import DeltaTable
 
# If running on Databricks, SparkSession is already available as `spark`
# If running locally, uncomment the lines below:
# spark = SparkSession.builder \
#     .appName("SCD2_Demo") \
#     .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
#     .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
#     .getOrCreate()
 
print("✅ Imports done.")

In [0]:
source_schema = StructType([
    StructField("customer_id", IntegerType(),  False),
    StructField("name",        StringType(),   True),
    StructField("city",        StringType(),   True),
    StructField("updated_at",  StringType(),   True),   # we'll cast to date below
])

In [0]:
# CELL 2 — CREATE SOURCE TABLE (Raw / Staging data)
# -----------------------------------------------------------------------------
# This simulates your source system data (e.g., from a CRM or operational DB).
# Think of this as the data that comes in today.
 
source_data = [
    (1, "Alice",   "Delhi",   "2026-01-01"),
    (2, "Bob",     "Mumbai",  "2026-01-01"),
    (3, "Charlie", "Chennai", "2026-01-01"),
    (4, "Diana",   "Pune",    "2026-01-01"),
    (5, "Eve",     "Kolkata", "2026-01-01"),
]
 
source_schema = StructType([
    StructField("customer_id", IntegerType(),  False),
    StructField("name",        StringType(),   True),
    StructField("city",        StringType(),   True),
    StructField("updated_at",  StringType(),   True),   # we'll cast to date below
])
 
source_df = spark.createDataFrame(source_data, schema=source_schema) \
                 .withColumn("updated_at", to_date(col("updated_at")))
 
print("✅ Source table created. Preview:")
source_df.show()

In [0]:
# -----------------------------------------------------------------------------
# CELL 3 — CREATE INITIAL DIMENSION TABLE (First Load)
# -----------------------------------------------------------------------------
# This is the very first load — all records are new, all marked as current.
# We write this as a Delta table which will be our SCD2 target going forward.
 
# Drop table if already exists (useful when re-running the notebook)
spark.sql("DROP TABLE IF EXISTS gold.dim_cust")
 
initial_dim_df = source_df \
    .withColumn("customer_sk", monotonically_increasing_id()) \
    .withColumn("start_date",  current_date()) \
    .withColumn("end_date",    lit(None).cast("date")) \
    .withColumn("is_current",  lit(1))
 
initial_dim_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.dim_cust")
 
print("✅ Initial dimension table created. Preview:")
spark.table("gold.dim_cust").show()
 

In [0]:
# -----------------------------------------------------------------------------
# CELL 4 — SIMULATE SOURCE DATA UPDATE
# -----------------------------------------------------------------------------
# Now pretend some time has passed and your source system has new data.
# Changes here:
#   - Alice:   city changed Delhi → Bangalore   (UPDATE)
#   - Bob:     no change                         (NO CHANGE)
#   - Charlie: city changed Chennai → Hyderabad  (UPDATE)
#   - Diana:   no change                         (NO CHANGE)
#   - Eve:     no change                         (NO CHANGE)
#   - Frank:   brand new customer               (INSERT)
#
# ✏️  You can change values here to test your own scenarios!
 
updated_source_data = [
    (1, "Alice",   "Bangalore", "2026-05-22"),   # city changed
    (2, "Bob",     "Mumbai",    "2026-01-01"),   # no change
    (3, "Charlie", "Hyderabad", "2026-05-22"),   # city changed
    (4, "Diana",   "Pune",      "2026-01-01"),   # no change
    (5, "Eve",     "kolMe",   "2026-01-02"),   # no change
    (6, "SCA",   "IND",    "2026-06-03"), 
    (6, "SCA",   "Utt",    "2026-06-03"),  # new customer
]
 
source_df = spark.createDataFrame(updated_source_data, schema=source_schema) \
                 .withColumn("updated_at", to_date(col("updated_at")))
 
print("✅ Updated source data ready. Preview:")
source_df.show()
 

In [0]:
# -----------------------------------------------------------------------------
# CELL 5 — DEDUPLICATE SOURCE DATA
# -----------------------------------------------------------------------------
# If source has multiple rows for the same customer, keep only the latest one.
# This avoids inserting duplicate active rows into the dim table.
 
w = Window.partitionBy("customer_id").orderBy(col("updated_at").desc())
 
source_df = source_df \
    .withColumn("rn", row_number().over(w)) \
    .filter(col("rn") == 1) \
    .drop("rn")
 
print("✅ Source deduplicated. Preview:")
source_df.show()
 
 

In [0]:
# -----------------------------------------------------------------------------
# CELL 6 — STEP A: EXPIRE CHANGED ROWS (SCD2 Core)
# -----------------------------------------------------------------------------
# For every customer whose city or name has changed:
#   → Set is_current = 0
#   → Set end_date   = today's date
# Unchanged rows are left untouched.
 
target = DeltaTable.forName(spark, "gold.dim_cust")
 
target.alias("tgt").merge(
    source_df.alias("s"),
    "tgt.customer_id = s.customer_id AND tgt.is_current = 1"
).whenMatchedUpdate(
    condition="""
        tgt.city IS DISTINCT FROM s.city
        OR tgt.name IS DISTINCT FROM s.name
    """,
    set={
        "is_current": lit(0),
        "end_date":   current_date()
    }
).execute()
 
print("✅ Step A done — changed rows expired. Current state of dim table:")
spark.table("gold.dim_cust").orderBy("customer_id").show()
 
 

In [0]:
# -----------------------------------------------------------------------------
# CELL 7 — STEP B: IDENTIFY ROWS TO INSERT
# -----------------------------------------------------------------------------
# After expiring old rows, we need to insert new versions.
# We find source rows that have NO active match in the dim table.
# This captures:
#   → Changed customers (their old row was just expired → no active row now)
#   → Brand new customers (never existed in dim table)
 
active_dim_df = spark.table("gold.dim_cust") \
    .filter(col("is_current") == 1) \
    .select("customer_id")
 
print("✅ Currently active customers in dim table:")
active_dim_df.show()
 
# left_anti: keep source rows that do NOT have a match in active_dim_df
final_df = source_df.join(active_dim_df, on="customer_id", how="left_anti")
 
print("✅ Rows to be inserted (new + changed):")
final_df.show()
 

In [0]:
# -----------------------------------------------------------------------------
# CELL 8 — STEP B: ADD SCD2 COLUMNS AND INSERT
# -----------------------------------------------------------------------------
# Add all required SCD2 metadata columns to the rows being inserted.
 
final_df = final_df \
    .withColumn("customer_sk", monotonically_increasing_id()) \
    .withColumn("start_date",  current_date()) \
    .withColumn("end_date",    lit(None).cast("date")) \
    .withColumn("is_current",  lit(1))
 
print("✅ New rows ready to insert:")
final_df.show()
 
# Append to Delta table
final_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("gold.dim_cust")
 
print("✅ New rows inserted successfully.")
 

In [0]:
# -----------------------------------------------------------------------------
# CELL 9 — FINAL RESULT
# -----------------------------------------------------------------------------
# View the complete SCD2 dimension table after processing.
# You should see:
#   → Expired rows  : is_current = 0, end_date = today
#   → New rows      : is_current = 1, end_date = null
#   → Unchanged rows: is_current = 1, end_date = null (untouched)
 
print("✅ Final SCD2 Dimension Table:")
spark.table("gold.dim_cust") \
    .orderBy("customer_id", "start_date") \
    .show(truncate=False)
 

In [0]:
# -----------------------------------------------------------------------------
# CELL 10 — VERIFY: ONLY ONE ACTIVE ROW PER CUSTOMER
# -----------------------------------------------------------------------------
# A key SCD2 rule: each customer should have exactly ONE row where is_current = 1.
# This check confirms data integrity.
 
from pyspark.sql.functions import count, when
 
check_df = spark.table("gold.dim_cust") \
    .groupBy("customer_id") \
    .agg(
        count(when(col("is_current") == 1, 1)).alias("active_rows")
    ) \
    .filter(col("active_rows") > 1)
 
if check_df.count() == 0:
    print("✅ Data quality check PASSED — every customer has exactly 1 active row.")
else:
    print("❌ Data quality check FAILED — some customers have multiple active rows:")
    check_df.show()

In [0]:
%sql
select * from gold.dim_cust